# PATCHR Server on Google Colab

Run the PATCHR inpainting server on Colab's free GPU and get a public URL to connect from **PATCHR-Studio**.

**Steps:**
1. Go to **Runtime > Change runtime type > GPU** (T4)
2. Run all cells below in order
3. Copy the public URL into PATCHR-Studio

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install PATCHR

In [ ]:
!git clone https://github.com/DeepFoldProtein/patchr.git
%cd patchr
!pip install -e .[cuda] -q

## 3. Start server

In [ ]:
import subprocess
import time

server_process = subprocess.Popen(
    ["python", "-m", "boltz.server", "--port", "31212", "--device-id", "0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("Starting PATCHR server...")
time.sleep(10)

import requests
try:
    r = requests.get("http://localhost:31212/api/v1/health")
    print("Server is ready:", r.json()["status"])
except Exception:
    print("Server is still loading, wait a moment...")

## 4. Create public tunnel

Choose **one** of the options below. No account or token required for any of them.

If a tunnel domain is blocked on your network, try a different option.

### Option A: Cloudflare Tunnel (`*.trycloudflare.com`)

In [ ]:
import subprocess, threading, re, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:31212"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)

tunnel_url = None
def read_output():
    global tunnel_url
    for line in tunnel_process.stderr:
        match = re.search(r"(https://[\w-]+\.trycloudflare\.com)", line.decode("utf-8", errors="ignore"))
        if match:
            tunnel_url = match.group(1)
            break

t = threading.Thread(target=read_output, daemon=True)
t.start()
t.join(timeout=15)

if tunnel_url:
    print("=" * 60)
    print(f"  Public URL: {tunnel_url}")
    print(f"  Copy this URL into PATCHR-Studio")
    print("=" * 60)
else:
    print("Failed to get tunnel URL. Try Option B or C.")

### Option B: localhost.run (`*.lhr.life`)

In [ ]:
import subprocess, re

tunnel_process = subprocess.Popen(
    ["ssh", "-o", "StrictHostKeyChecking=no", "-R", "80:localhost:31212", "nokey@localhost.run"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)

# localhost.run prints the URL to stdout
for line in tunnel_process.stdout:
    line = line.decode("utf-8", errors="ignore")
    match = re.search(r"(https://[\w]+\.lhr\.life)", line)
    if match:
        print("=" * 60)
        print(f"  Public URL: {match.group(1)}")
        print(f"  Copy this URL into PATCHR-Studio")
        print("=" * 60)
        break

### Option C: bore (`bore.pub`)

In [ ]:
import subprocess, re

# Install bore
!wget -q https://github.com/ekzhang/bore/releases/latest/download/bore-v0.5.2-x86_64-unknown-linux-musl.tar.gz
!tar -xzf bore-v0.5.2-x86_64-unknown-linux-musl.tar.gz

tunnel_process = subprocess.Popen(
    ["./bore", "local", "31212", "--to", "bore.pub"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)

# bore prints the URL to stderr
for line in tunnel_process.stderr:
    line = line.decode("utf-8", errors="ignore")
    match = re.search(r"bore\.pub:(\d+)", line)
    if match:
        port = match.group(1)
        print("=" * 60)
        print(f"  Public URL: http://bore.pub:{port}")
        print(f"  Copy this URL into PATCHR-Studio")
        print("=" * 60)
        break

## 5. Stop server

Run this cell when you are done.

In [ ]:
tunnel_process.terminate()
server_process.terminate()
print("Server stopped.")